# RAR Per-Seed Campaign — Clean Runner

Runs the RAR experiment **one seed at a time**. Each seed writes its own
`pilot_seed_<seed>.json`, so a Colab disconnect never wipes finished work.

**Key handling:** loads automatically from **Colab Secrets** (the 🔑 icon).
Secret name must be `OPENROUTER_API_KEY` with *Notebook access* ON.

**Drive backup is optional** — if mounting fails, the run continues on local disk.

**Safety guard:** if the key is ever missing, the code *raises* instead of
silently fabricating fake results.

Run cells top to bottom. Don't start the seed loop until **PRE-FLIGHT** passes.

## 1 · Clone repo & install deps

In [ ]:
import os

REPO_DIR = '/content/recursive-autonomy-research'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tahir-yamin/recursive-autonomy-research.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!pip install -q torch numpy scipy scikit-learn networkx aiohttp matplotlib

## 2 · Load API key from Colab Secrets

One-time setup: 🔑 icon → **+ Add new secret** → name `OPENROUTER_API_KEY`,
paste your key, toggle **Notebook access** ON. Then just run this cell.

In [ ]:
import os

key = None
try:
    from google.colab import userdata
    key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    print('Could not read Colab Secret:', e)

if not key:
    raise ValueError(
        '\n>>> No key found. Open the 🔑 panel, add OPENROUTER_API_KEY, '
        'and turn ON Notebook access for it, then re-run this cell.'
    )

os.environ['OPENROUTER_API_KEY'] = key
os.environ['RAR_CYCLES']         = '60'
# Model is auto-detected in the PRE-FLIGHT cell (free slugs rotate often).
print(f'Key loaded (ends ...{key[-6:]})')
print('Model will be auto-selected in the PRE-FLIGHT cell.')

## 3 · (Optional) Mount Google Drive for backup

If mounting fails, this cell just warns and the run continues on local disk.

In [ ]:
DRIVE_BACKUP = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP = '/content/drive/MyDrive/RAR_results'
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    print('Drive backup ON ->', DRIVE_BACKUP)
except Exception as e:
    DRIVE_BACKUP = None
    print('Drive mount skipped (continuing on local disk only):', e)

## 4 · PRE-FLIGHT — auto-detect a working free model

Free `:free` slugs rotate constantly on OpenRouter. This cell probes a list of
current free candidates with one tiny call each and **locks onto the first that
responds**, setting `OPENROUTER_MODEL` for the whole run.

**Do not proceed if every candidate fails** — fix the key/quota first.

In [ ]:
import sys, os
os.chdir('/content/recursive-autonomy-research')
if 'run_pilot_experiment' in sys.modules:
    del sys.modules['run_pilot_experiment']
import run_pilot_experiment as rpe

# Candidate free models, in preference order. First one that answers wins.
# openai/gpt-oss-20b:free confirmed working Jun 2026 on this account; others
# returned 404 "paid only" or 429 rate-limit. Reorder/add as the free tier shifts.
CANDIDATES = [
    'openai/gpt-oss-20b:free',
    'meta-llama/llama-3.3-70b-instruct:free',
    'google/gemma-3-12b-it:free',
    'deepseek/deepseek-chat-v3:free',
    'qwen/qwen-2.5-72b-instruct:free',
]

chosen = None
for slug in CANDIDATES:
    os.environ['OPENROUTER_MODEL'] = slug
    try:
        resp = await rpe.call_llm('Reply with the single word: OK')
    except Exception as e:
        print(f'  ✗ {slug:45s} error: {str(e)[:80]}')
        continue
    if resp:
        chosen = slug
        print(f'  ✓ {slug:45s} -> {resp[:40]!r}')
        break
    else:
        print(f'  ✗ {slug:45s} empty response')

if not chosen:
    raise RuntimeError(
        '>>> PRE-FLIGHT FAILED: no candidate free model responded.\n'
        'Check key validity / daily quota, or add a known-good slug to CANDIDATES.\n'
        'Browse current free models at https://openrouter.ai/models?max_price=0'
    )

os.environ['OPENROUTER_MODEL'] = chosen
print(f'\nPRE-FLIGHT OK — using model: {chosen}')
print('This model is now locked for the whole campaign.')

## 5 · Run seeds (one at a time) — choose a benchmark

Set **`DATASET`** at the top of the next cell to one of: `manifold` (original),
`digits` (real, 10-class), or `synth_tuned` (4-class, above chance).

To run all benchmarks: pick a `DATASET`, run cells 5 → 6 → 7, then change
`DATASET` and repeat. Per-seed files are namespaced (`pilot_seed_<seed>_<DATASET>.json`)
so the runs never collide, and completed seeds auto-skip on resume.

In [ ]:
import os, sys, shutil, glob
os.chdir('/content/recursive-autonomy-research')

# ====== CHOOSE THE BENCHMARK ======
# 'manifold'    = original near-chance synthetic task (already reported)
# 'digits'      = real sklearn handwritten digits (10-class, ceiling ~96%)
# 'synth_tuned' = tunable make_classification (4-class, ceiling ~77%)
DATASET = 'digits'          # <-- change to 'synth_tuned' for the second new run
os.environ['RAR_DATASET'] = DATASET

seeds_to_run = [42, 7, 13, 23, 88, 99, 101, 107, 113, 127]

if not os.environ.get('OPENROUTER_API_KEY'):
    raise ValueError('Key missing — re-run the key cell.')

for mod in ('run_pilot_experiment', 'run_deep_learning_harness'):
    if mod in sys.modules:
        del sys.modules[mod]
import run_pilot_experiment as rpe   # re-imported so RAR_DATASET takes effect

# restore any seed files for THIS dataset already backed up to Drive
if DRIVE_BACKUP and os.path.exists(DRIVE_BACKUP):
    for f in glob.glob(os.path.join(DRIVE_BACKUP, f'pilot_seed_*_{DATASET}.json')):
        dst = os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy2(f, dst); print('Restored', dst, 'from Drive')

for seed in seeds_to_run:
    out_file = f'pilot_seed_{seed}_{DATASET}.json'
    if os.path.exists(out_file):
        print(f'Seed {seed} [{DATASET}]: already done, skipping.')
        continue
    print(f'\n===== STARTING SEED {seed} [{DATASET}] =====')
    os.environ['RAR_OUTPUT_FILE'] = out_file
    rpe.SEEDS = [seed]
    rpe.CYCLES = int(os.environ.get('RAR_CYCLES', '60'))
    await rpe.execute_campaign()
    if os.path.exists(out_file):
        print(f'Seed {seed}: DONE -> {out_file} ({os.path.getsize(out_file):,} bytes)')
        if DRIVE_BACKUP:
            shutil.copy2(out_file, os.path.join(DRIVE_BACKUP, out_file))
            print('   backed up to Drive')
    else:
        print(f'WARNING: seed {seed} produced no file!')

print(f'\n===== ALL SEEDS COMPLETE for [{DATASET}] =====')

## 6 · Merge seeds → final pilot_results.json (real Wilcoxon p)

In [ ]:
import os, glob, shutil
os.chdir('/content/recursive-autonomy-research')

# uses DATASET set in the seed-loop cell above
if DRIVE_BACKUP and os.path.exists(DRIVE_BACKUP):
    for f in glob.glob(os.path.join(DRIVE_BACKUP, f'pilot_seed_*_{DATASET}.json')):
        dst = os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy2(f, dst)

seed_files = sorted(glob.glob(f'pilot_seed_*_{DATASET}.json'))
print(f'{len(seed_files)} seed files for [{DATASET}]:', [os.path.basename(f) for f in seed_files])

if len(seed_files) < 2:
    print('Need >=2 seeds to merge.')
else:
    !python merge_seeds.py {DATASET}
    merged = f'pilot_results_{DATASET}.json'
    if DRIVE_BACKUP and os.path.exists(merged):
        shutil.copy2(merged, os.path.join(DRIVE_BACKUP, merged))
        print('Merged result backed up to Drive ->', merged)

## 7 · Sanity check — the honest numbers

In [ ]:
import json
with open(f'pilot_results_{DATASET}.json') as f:
    r = json.load(f)
print('Dataset:', r.get('dataset_tag'))
print('Seeds :', r['SEEDS'])
print('Cycles:', r['CYCLES'])
print('Wilcoxon p (RAR > Baseline):', r['wilcoxon_p_value_RAR_vs_Baseline'])
print()
for cond in ['stateless_baseline', 'vector_rag', 'rar_compressed']:
    d = r['data']['conditions'][cond]
    ta = d['test_accuracies']; nt = d['net_tokens']
    print(f'{cond:20s} test_acc={sum(ta)/len(ta):.4f}  net_tokens={sum(nt)/len(nt):,.0f}')